# Data Pipeline Notebook

This notebook prepares VN-Index and Google Trends SVI data for downstream volatility modeling in `note.ipynb`.

Outputs are saved to `outputs/`:
- `data_daily.csv`
- `data_weekly.csv`

## 1) Install and Import Dependencies

In [7]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = [
    "numpy",
    "pandas",
    "statsmodels",
    "scipy",
    "matplotlib",
    "vnstock",
    "pytrends",
]

def ensure_package(pkg_name: str) -> None:
    try:
        importlib.import_module(pkg_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])

for pkg in REQUIRED_PACKAGES:
    ensure_package(pkg)

import json
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import vnstock as vns
from pandas.errors import EmptyDataError
from pytrends.exceptions import TooManyRequestsError
from pytrends.request import TrendReq

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Python: 3.11.4
pandas: 3.0.0
numpy: 2.4.1


## 2) Create Configuration and Constants

In [8]:
@dataclass
class DataConfig:
    symbol: str = "VNINDEX"
    fallback_symbols: Tuple[str, ...] = ("VN-INDEX", "VNI")
    price_source: str = "VCI"
    price_csv_path: Optional[str] = None

    start_date: str = "2009-12-01"
    end_date: str = "2025-12-31"

    keywords: Tuple[str, ...] = ("chung khoan", "VN-Index", "co phieu", "dau tu")
    geo: str = "VN"
    hl: str = "en-US"
    tz: int = 420

    trend_window_days: int = 240
    trend_overlap_days: int = 14
    trend_sleep_seconds: float = 8.0
    trend_max_retries: int = 5
    trend_backoff_base_seconds: float = 5.0
    trend_backoff_max_seconds: float = 10.0
    trend_cache_dir: str = "outputs/trends_cache"

    api_sleep_seconds: float = 1.5
    api_max_retries: int = 4
    api_backoff_base_seconds: float = 2.0
    api_backoff_max_seconds: float = 30.0

    smoothing_window: int = 3
    lag_order: int = 5
    output_dir: str = "outputs"
    daily_filename: str = "data_daily"
    weekly_filename: str = "data_weekly"

CONFIG = DataConfig()
OUTPUT_DIR = Path(CONFIG.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Config loaded")
print(json.dumps(asdict(CONFIG), indent=2))

Config loaded
{
  "symbol": "VNINDEX",
  "fallback_symbols": [
    "VN-INDEX",
    "VNI"
  ],
  "price_source": "VCI",
  "price_csv_path": null,
  "start_date": "2009-12-01",
  "end_date": "2025-12-31",
  "keywords": [
    "chung khoan",
    "VN-Index",
    "co phieu",
    "dau tu"
  ],
  "geo": "VN",
  "hl": "en-US",
  "tz": 420,
  "trend_window_days": 240,
  "trend_overlap_days": 14,
  "trend_sleep_seconds": 8.0,
  "trend_max_retries": 5,
  "trend_backoff_base_seconds": 5.0,
  "trend_backoff_max_seconds": 10.0,
  "trend_cache_dir": "outputs/trends_cache",
  "api_sleep_seconds": 1.5,
  "api_max_retries": 4,
  "api_backoff_base_seconds": 2.0,
  "api_backoff_max_seconds": 30.0,
  "smoothing_window": 3,
  "lag_order": 5,
  "output_dir": "outputs",
  "daily_filename": "data_daily",
  "weekly_filename": "data_weekly"
}


## 3) Implement Core Functions

In [9]:
def get_active_date_range(config: DataConfig) -> Tuple[pd.Timestamp, pd.Timestamp]:
    return pd.Timestamp(config.start_date), pd.Timestamp(config.end_date)


def _sleep_with_backoff(attempt: int, base_seconds: float, max_seconds: float) -> None:
    wait_seconds = min(base_seconds * (2 ** max(0, attempt - 1)), max_seconds)
    time.sleep(wait_seconds)


def _api_call_with_retry(call_fn, *, call_name: str, max_retries: int, base_backoff: float, max_backoff: float):
    last_error: Optional[Exception] = None
    for attempt in range(1, max_retries + 1):
        try:
            return call_fn()
        except Exception as exc:
            last_error = exc
            if attempt == max_retries:
                break
            _sleep_with_backoff(attempt, base_backoff, max_backoff)
    raise RuntimeError(f"API call failed after retries: {call_name}") from last_error


def _normalize_vnstock_history(df: pd.DataFrame, config: DataConfig, symbol_tag: str) -> pd.DataFrame:
    lowered = {c.lower(): c for c in df.columns}
    date_col = lowered.get("time") or lowered.get("date") or lowered.get("datetime")
    close_col = lowered.get("close") or lowered.get("adj close") or lowered.get("adj_close")
    if date_col is None or close_col is None:
        raise ValueError(f"Unexpected vnstock columns: {list(df.columns)}")

    out = df[[date_col, close_col]].copy()
    out.columns = ["date", "close"]
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out.dropna(subset=["date", "close"])
    out = out.set_index("date").sort_index()

    start_ts, end_ts = get_active_date_range(config)
    out = out.loc[(out.index >= start_ts) & (out.index <= end_ts)]
    out["symbol_used"] = symbol_tag
    return out


def _download_from_vnstock(symbol: str, config: DataConfig) -> pd.DataFrame:
    start_ts, end_ts = get_active_date_range(config)
    start_str = start_ts.strftime("%Y-%m-%d")
    end_str = end_ts.strftime("%Y-%m-%d")

    def _download_legacy() -> pd.DataFrame:
        return vns.stock_historical_data(
            symbol=symbol,
            start_date=start_str,
            end_date=end_str,
            resolution="1D",
            type="index",
            source=config.price_source,
            beautify=False,
            decor=False,
        )

    def _download_legacy_fallback() -> pd.DataFrame:
        return vns.stock_historical_data(
            symbol=symbol,
            start_date=start_str,
            end_date=end_str,
            resolution="1D",
            type="index",
        )

    def _download_new_api() -> pd.DataFrame:
        client = vns.Vnstock()
        q = client.stock(symbol=symbol, source=config.price_source).quote
        return q.history(start=start_str, end=end_str, interval="1D")

    if hasattr(vns, "stock_historical_data"):
        try:
            hist = _api_call_with_retry(
                _download_legacy,
                call_name=f"vnstock.stock_historical_data({symbol})",
                max_retries=config.api_max_retries,
                base_backoff=config.api_backoff_base_seconds,
                max_backoff=config.api_backoff_max_seconds,
            )
            time.sleep(config.api_sleep_seconds)
            if isinstance(hist, pd.DataFrame) and not hist.empty:
                return hist
        except TypeError:
            hist = _api_call_with_retry(
                _download_legacy_fallback,
                call_name=f"vnstock.stock_historical_data fallback({symbol})",
                max_retries=config.api_max_retries,
                base_backoff=config.api_backoff_base_seconds,
                max_backoff=config.api_backoff_max_seconds,
            )
            time.sleep(config.api_sleep_seconds)
            if isinstance(hist, pd.DataFrame) and not hist.empty:
                return hist

    if hasattr(vns, "Vnstock"):
        hist = _api_call_with_retry(
            _download_new_api,
            call_name=f"vnstock.Vnstock().quote.history({symbol})",
            max_retries=config.api_max_retries,
            base_backoff=config.api_backoff_base_seconds,
            max_backoff=config.api_backoff_max_seconds,
        )
        time.sleep(config.api_sleep_seconds)
        if isinstance(hist, pd.DataFrame) and not hist.empty:
            return hist

    return pd.DataFrame()


def _load_prices_from_csv(csv_path: Path, config: DataConfig) -> pd.DataFrame:
    raw = pd.read_csv(csv_path)
    lowered = {c.lower(): c for c in raw.columns}

    date_col = lowered.get("date") or lowered.get("datetime")
    close_col = lowered.get("close") or lowered.get("adj close") or lowered.get("adj_close")
    if date_col is None or close_col is None:
        raise ValueError("CSV must contain date/datetime and close columns")

    out = raw[[date_col, close_col]].copy()
    out.columns = ["date", "close"]
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out.dropna(subset=["date", "close"])
    out = out.set_index("date").sort_index()

    start_ts, end_ts = get_active_date_range(config)
    out = out.loc[(out.index >= start_ts) & (out.index <= end_ts)]
    out["symbol_used"] = "LOCAL_CSV"
    return out


def fetch_vnindex_prices(config: DataConfig) -> pd.DataFrame:
    if config.price_csv_path:
        csv_path = Path(config.price_csv_path)
        if not csv_path.exists():
            raise FileNotFoundError(f"Configured price_csv_path not found: {csv_path}")
        return _load_prices_from_csv(csv_path, config)

    candidates = [config.symbol] + [s for s in config.fallback_symbols if s != config.symbol]
    errors: List[str] = []

    for symbol in candidates:
        try:
            df = _download_from_vnstock(symbol, config)
            if df.empty:
                errors.append(f"{symbol}: empty dataframe")
                continue

            out = _normalize_vnstock_history(df, config, symbol)
            if out.empty:
                errors.append(f"{symbol}: close series empty after cleaning")
                continue
            return out
        except Exception as exc:
            errors.append(f"{symbol}: {type(exc).__name__} - {exc}")

    details = " | ".join(errors)
    raise ValueError(
        "No VN-Index data returned from vnstock. "
        f"Tried symbols={candidates}. Details: {details}. "
        "Check vnstock source/symbol or provide price_csv_path in config to use local CSV."
    )


def build_date_windows(start_ts: pd.Timestamp, end_ts: pd.Timestamp, window_days: int, overlap_days: int) -> List[Tuple[pd.Timestamp, pd.Timestamp]]:
    if overlap_days >= window_days:
        raise ValueError("overlap_days must be smaller than window_days")

    windows: List[Tuple[pd.Timestamp, pd.Timestamp]] = []
    step = window_days - overlap_days
    current_start = start_ts

    while current_start <= end_ts:
        current_end = min(current_start + pd.Timedelta(days=window_days - 1), end_ts)
        windows.append((current_start, current_end))
        if current_end >= end_ts:
            break
        current_start = current_start + pd.Timedelta(days=step)

    return windows


def _trend_cache_path(config: DataConfig, keyword: str, start_ts: pd.Timestamp, end_ts: pd.Timestamp) -> Path:
    cache_dir = Path(config.trend_cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    safe_keyword = "".join(ch if ch.isalnum() else "_" for ch in keyword)
    filename = f"{safe_keyword}_{start_ts.strftime('%Y%m%d')}_{end_ts.strftime('%Y%m%d')}.csv"
    return cache_dir / filename


def fetch_trends_window(pytrends: TrendReq, keyword: str, start_ts: pd.Timestamp, end_ts: pd.Timestamp, geo: str, config: DataConfig) -> pd.Series:
    cache_path = _trend_cache_path(config, keyword, start_ts, end_ts)
    if cache_path.exists():
        cached = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        if not cached.empty and keyword in cached.columns:
            series = cached[keyword].astype(float)
            series.index = pd.to_datetime(series.index).tz_localize(None)
            series.name = keyword
            return series

    timeframe = f"{start_ts.strftime('%Y-%m-%d')} {end_ts.strftime('%Y-%m-%d')}"
    attempts = max(1, config.trend_max_retries)

    for attempt in range(1, attempts + 1):
        try:
            pytrends.build_payload([keyword], timeframe=timeframe, geo=geo)
            data = pytrends.interest_over_time()
            time.sleep(config.trend_sleep_seconds)

            if data.empty or keyword not in data.columns:
                return pd.Series(dtype=float, name=keyword)

            series = data[keyword].astype(float)
            series.index = pd.to_datetime(series.index).tz_localize(None)
            series.name = keyword
            pd.DataFrame({keyword: series}).to_csv(cache_path)
            return series
        except TooManyRequestsError:
            if attempt == attempts:
                print(f"429 error, please try again later.")
                sys.exit(0)
            wait_seconds = min(
                config.trend_backoff_base_seconds * (2 ** (attempt - 1)),
                config.trend_backoff_max_seconds,
            )
            print(f"429 for {keyword} [{timeframe}] attempt {attempt}/{attempts}, waiting {wait_seconds:.1f}s")
            time.sleep(wait_seconds)

    return pd.Series(dtype=float, name=keyword)


def stitch_overlapping_series(chunks: List[pd.Series]) -> pd.Series:
    if not chunks:
        return pd.Series(dtype=float)

    stitched = chunks[0].copy()
    for nxt in chunks[1:]:
        overlap_idx = stitched.index.intersection(nxt.index)
        if len(overlap_idx) > 0:
            prev_mean = stitched.loc[overlap_idx].replace(0, np.nan).mean()
            next_mean = nxt.loc[overlap_idx].replace(0, np.nan).mean()
            if pd.notna(prev_mean) and pd.notna(next_mean) and next_mean > 0:
                factor = prev_mean / next_mean
            else:
                factor = 1.0
        else:
            factor = 1.0

        scaled_next = nxt * factor
        stitched = pd.concat([stitched, scaled_next[~scaled_next.index.isin(stitched.index)]])
        stitched = stitched.sort_index()

    return stitched


def fetch_and_stitch_keyword(config: DataConfig, keyword: str) -> pd.Series:
    start_ts, end_ts = get_active_date_range(config)
    windows = build_date_windows(start_ts, end_ts, config.trend_window_days, config.trend_overlap_days)
    pytrends = TrendReq(hl=config.hl, tz=config.tz)

    chunks: List[pd.Series] = []
    for w_start, w_end in windows:
        series = fetch_trends_window(pytrends, keyword, w_start, w_end, config.geo, config)
        if not series.empty:
            chunks.append(series)

    stitched = stitch_overlapping_series(chunks)
    stitched.name = keyword
    return stitched


def build_svi_composite(config: DataConfig) -> pd.DataFrame:
    keyword_series: Dict[str, pd.Series] = {}
    for kw in config.keywords:
        keyword_series[kw] = fetch_and_stitch_keyword(config, kw)

    svi_df = pd.DataFrame(keyword_series).sort_index()
    if svi_df.empty:
        raise ValueError("No SVI data collected. Check keywords, geo, or network access.")

    svi_norm = (svi_df - svi_df.mean()) / svi_df.std(ddof=0)
    svi_smooth = svi_norm.rolling(config.smoothing_window, min_periods=1).mean()
    svi_composite = svi_smooth.mean(axis=1)

    result = pd.DataFrame(
        {
            "svi_raw_mean": svi_df.mean(axis=1),
            "svi_norm_composite": svi_norm.mean(axis=1),
            "svi_sma_composite": svi_composite,
        }
    )
    return result


def build_daily_dataset(price_df: pd.DataFrame, svi_df: pd.DataFrame, config: DataConfig) -> pd.DataFrame:
    df = price_df.join(svi_df, how="inner").copy()
    df["return"] = np.log(df["close"] / df["close"].shift(1)) * 100.0
    df["rv_proxy"] = df["return"] ** 2

    for lag in range(1, config.lag_order + 1):
        df[f"svi_lag{lag}"] = df["svi_sma_composite"].shift(lag)

    df = df.dropna().sort_index()
    df.index.name = "date"
    return df


def build_weekly_dataset(daily_df: pd.DataFrame, config: DataConfig) -> pd.DataFrame:
    w = daily_df.resample("W-FRI").agg({
        "close": "last",
        "svi_sma_composite": "mean",
        "rv_proxy": "sum" 
    }).dropna()

    w["return"] = np.log(w["close"] / w["close"].shift(1)) * 100.0
    
    for lag in range(1, config.lag_order + 1):
        w[f"svi_lag{lag}"] = w["svi_sma_composite"].shift(lag)

    return w.dropna().sort_index()


def run_quality_checks(df: pd.DataFrame, name: str) -> Dict[str, object]:
    report = {
        "name": name,
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "null_values": int(df.isnull().sum().sum()),
        "duplicated_index": int(df.index.duplicated().sum()),
        "start": str(df.index.min()) if len(df) else None,
        "end": str(df.index.max()) if len(df) else None,
    }
    return report


def save_dataframe(df: pd.DataFrame, base_path: Path) -> Path:
    out_path = base_path.with_suffix(".csv")
    df.to_csv(out_path, index=True)
    return out_path

## 4) Build a Minimal Execution Pipeline

In [10]:
def _read_existing_daily(config: DataConfig) -> pd.DataFrame:
    daily_path = OUTPUT_DIR / f"{config.daily_filename}.csv"
    if not daily_path.exists():
        return pd.DataFrame()

    try:
        existing = pd.read_csv(daily_path, index_col=0, parse_dates=True)
    except EmptyDataError:
        return pd.DataFrame()

    if existing.empty:
        return pd.DataFrame()

    existing.index = pd.to_datetime(existing.index).tz_localize(None)
    existing.index.name = "date"
    return existing.sort_index()


def _get_resume_start(config: DataConfig, existing_daily: pd.DataFrame) -> Tuple[pd.Timestamp, pd.Timestamp, bool]:
    start_ts, end_ts = get_active_date_range(config)
    if existing_daily.empty:
        return start_ts, start_ts, False

    last_available = existing_daily.index.max().normalize()
    if last_available >= end_ts:
        return start_ts, end_ts + pd.Timedelta(days=1), True

    warmup_days = max(45, config.lag_order + 10)
    fetch_start = max(start_ts, last_available - pd.Timedelta(days=warmup_days))
    return start_ts, fetch_start, False


def build_and_export_datasets(config: DataConfig) -> Dict[str, object]:
    start_ts, end_ts = get_active_date_range(config)
    existing_daily = _read_existing_daily(config)
    full_start, fetch_start, already_complete = _get_resume_start(config, existing_daily)

    if already_complete:
        daily = existing_daily.loc[(existing_daily.index >= full_start) & (existing_daily.index <= end_ts)].copy()
        weekly = build_weekly_dataset(daily, config)
    else:
        run_config = DataConfig(**{**asdict(config), "start_date": fetch_start.strftime("%Y-%m-%d")})
        prices = fetch_vnindex_prices(run_config)
        svi = build_svi_composite(run_config)
        daily_new = build_daily_dataset(prices, svi, run_config)

        if existing_daily.empty:
            daily = daily_new.copy()
        else:
            keep_old = existing_daily.loc[existing_daily.index < fetch_start].copy()
            daily = pd.concat([keep_old, daily_new], axis=0)
            daily = daily[~daily.index.duplicated(keep="last")].sort_index()

        daily = daily.loc[(daily.index >= full_start) & (daily.index <= end_ts)].copy()
        weekly = build_weekly_dataset(daily, config)

    daily_path = save_dataframe(daily, OUTPUT_DIR / config.daily_filename)
    weekly_path = save_dataframe(weekly, OUTPUT_DIR / config.weekly_filename)

    return {
        "daily": daily,
        "weekly": weekly,
        "daily_path": daily_path,
        "weekly_path": weekly_path,
    }

## 5) Add Sanity Checks and Assertions

In [11]:
def validate_output_schema(df: pd.DataFrame, lag_order: int) -> None:
    required = ["close", "return", "rv_proxy", "svi_sma_composite"] + [f"svi_lag{i}" for i in range(1, lag_order + 1)]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"Missing columns: {missing}"
    assert not df.index.duplicated().any(), "Duplicated index values found"
    assert df.isnull().sum().sum() == 0, "Null values found in final dataset"


def run_validation_bundle(results: Dict[str, object], config: DataConfig) -> Dict[str, Dict[str, object]]:
    daily = results["daily"]
    weekly = results["weekly"]

    validate_output_schema(daily, config.lag_order)
    validate_output_schema(weekly, config.lag_order)

    reports = {
        "daily": run_quality_checks(daily, "daily"),
        "weekly": run_quality_checks(weekly, "weekly"),
    }
    return reports

## 6) Run End-to-End Execution

In [12]:
RUN_CONFIG = DataConfig()

results = build_and_export_datasets(RUN_CONFIG)
reports = run_validation_bundle(results, RUN_CONFIG)

print("Daily output:", results["daily_path"])
print("Weekly output:", results["weekly_path"])
print("Quality reports:")
print(json.dumps(reports, indent=2))

display(results["daily"].head())
display(results["weekly"].head())

2026-03-30 21:12:42 - vnstock.common.data - INFO - Not a stock. Company and finance data unavailable.


Daily output: outputs\data_daily.csv
Weekly output: outputs\data_weekly.csv
Quality reports:
{
  "daily": {
    "name": "daily",
    "rows": 4010,
    "columns": 12,
    "null_values": 0,
    "duplicated_index": 0,
    "start": "2009-12-09 00:00:00",
    "end": "2025-12-31 00:00:00"
  },
  "weekly": {
    "name": "weekly",
    "rows": 825,
    "columns": 9,
    "null_values": 0,
    "duplicated_index": 0,
    "start": "2010-01-15 00:00:00",
    "end": "2026-01-02 00:00:00"
  }
}


,close,symbol_used,svi_raw_mean,svi_norm_composite,svi_sma_composite,return,rv_proxy,svi_lag1,svi_lag2,svi_lag3,svi_lag4,svi_lag5
date,,,,,,,,,,,,
2009-12-09,470.6,VNINDEX,51.00,2.520955,2.202735,-3.733100,13.936032,1.362296,2.295730,2.700856,2.968797,3.335031
2009-12-10,458.7,VNINDEX,34.25,1.093858,1.943076,-2.561207,6.559784,2.202735,1.362296,2.295730,2.700856,2.968797
2009-12-11,444.2,VNINDEX,43.00,1.947638,1.854150,-3.212149,10.317901,1.943076,2.202735,1.362296,2.295730,2.700856
2009-12-14,458.4,VNINDEX,46.50,2.148904,1.187870,3.146725,9.901881,1.854150,1.943076,2.202735,1.362296,2.295730
2009-12-15,459.4,VNINDEX,44.75,2.028432,1.516628,0.217912,0.047486,1.187870,1.854150,1.943076,2.202735,1.362296


,close,svi_sma_composite,rv_proxy,return,svi_lag1,svi_lag2,svi_lag3,svi_lag4,svi_lag5
date,,,,,,,,,
2010-01-15,505.4,2.544376,18.798895,-3.020789,2.220680,1.597532,1.873991,1.529631,1.999987
2010-01-22,477.6,1.713904,23.298704,-5.657663,2.544376,2.220680,1.597532,1.873991,1.529631
2010-01-29,482.0,1.720931,18.907997,0.917055,1.713904,2.544376,2.220680,1.597532,1.873991
2010-02-05,493.0,1.190573,10.943829,2.256506,1.720931,1.713904,2.544376,2.220680,1.597532
2010-02-12,507.0,0.478897,11.027302,2.800183,1.190573,1.720931,1.713904,2.544376,2.220680
